In [21]:
import pandas as pd
import numpy as np
import json
import re

# Load dataset using pandas
with open("jobs_v2.json", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(df.shape)

(585, 15)


In [22]:
df_before = df.copy()

In [23]:
# ---------- Column names ----------
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("&", "and", regex=False)
)

In [24]:
# ---------- Clean text ----------
exclude_cols = ["skills_and_tools", "experience_needed"]
text_cols = df.select_dtypes(include=["object", "string"]).columns
strip_cols = [c for c in text_cols if c not in exclude_cols]

for col in strip_cols:
    df[col] = df[col].str.strip()


In [25]:
# ---------- Missing values ----------
for col in strip_cols:
    df[col] = df[col].replace(["N/A", ""], np.nan)

for col in strip_cols:
    if col != "salary":
        df[col] = df[col].fillna("Unknown")

In [26]:
# ---------- Remove duplicates ----------
df = df.drop_duplicates(subset="url").reset_index(drop=True)

In [27]:
#  Arabic location translation ----------
arabic_location_map = {
    "هليوبوليس, القاهرة, مصر": "Heliopolis, Cairo, Egypt",
    "القاهرة الجديدة, القاهرة, مصر": "New Cairo, Cairo, Egypt",
}
df["location"] = df["location"].replace(arabic_location_map)

In [28]:
# ---------- Company size ----------
arabic_company_size = {
    "١١ - ٥٠ موظف": "11-50 employees",
    "٥٠١ - ١٠٠٠ موظف": "501-1000 employees",
}
df["company_size"] = df["company_size"].replace(arabic_company_size)

def parse_company_size(s):
    if pd.isna(s) or s == "Unknown":
        return np.nan, np.nan, np.nan

    s = str(s)

    more = re.search(r"more than\s+(\d+)", s, re.IGNORECASE)
    if more:
        v = int(more.group(1))
        return v, v, float(v)

    nums = re.findall(r"\d+", s)
    if len(nums) == 1:
        v = int(nums[0])
        return v, v, float(v)

    if len(nums) >= 2:
        lo, hi = int(nums[0]), int(nums[1])
        return lo, hi, (lo + hi) / 2

    return np.nan, np.nan, np.nan

# Keep the original company_size column, add min/max/avg alongside it
df[["company_size_min", "company_size_max", "company_size_avg"]] = (
    df["company_size"].apply(parse_company_size).apply(pd.Series)
)

median_size = df["company_size_avg"].median()
df["company_size_avg"] = df["company_size_avg"].fillna(median_size)
df["company_size_min"] = df["company_size_min"].fillna(median_size)
df["company_size_max"] = df["company_size_max"].fillna(median_size)

# Cast min/max/avg to integer (employees are whole numbers)
df["company_size_avg"] = df["company_size_avg"].round().astype(int)
df["company_size_min"] = df["company_size_min"].round().astype(int)
df["company_size_max"] = df["company_size_max"].round().astype(int)


In [29]:
from IPython.display import display

sample_idx = df_before.sample(5, random_state=42).index

print("Before:")
display(df_before.loc[sample_idx])

print("After:")
display(df.loc[sample_idx])

Before:


,Job Title,Company Name,Company Size,Location,Work Type,Work Setting,Experience Needed,Career Level,Education Level,Salary,Skills & Tools,Job Description,Job Requirements,Posted At,url
383,Senior Document Controller,EMS Solutions,11-50 employees,"6th of October, Giza, Egypt",Full Time,Hybrid,"[5, 10]",Manager,Not Specified,N/A,"[Business Administration, Document Controlling...",Responsibilities: Implement and maintain do...,Bachelor's degree in Business Administration ...,02/26/2026 17:26:44,https://wuzzuf.net/jobs/p/bfasvms4eet5-senior-...
422,Senior Software Engineering Manager-Java,Link Development,501-1000 employees,"Maadi, Cairo, Egypt",Full Time,Hybrid,"[10, 15]",Manager,Bachelor's Degree,N/A,"[Software Engineering, Java, Software Developm...",We are seeking a Senior Software Engineering ...,Proven experience leading teams delivering Ja...,03/09/2026 13:36:04,https://wuzzuf.net/jobs/p/f409jot9zrmg-senior-...
90,Senior Data Scientist,Egabi FSI,101-500 employees,"Sheraton, Cairo, Egypt",Full Time,On-site,"[3, above_min]",Experienced (Non-Manager),Bachelor's Degree,N/A,"[Computer Science, Data Mining, Data Science, ...","Collect, clean, and preprocess data from vario...","Bachelor’s degree in Computer Science, Data Sc...",03/17/2026 10:39:43,https://wuzzuf.net/jobs/p/o1a7fazksizo-senior-...
472,AI Developer (Outsource),Prometeon tyres group – Egypt,More than 1000 employees,"Alexandria, Egypt",Full Time,On-site,"[0, 2]",Entry Level (Junior Level / Fresh Grad),Bachelor's Degree,N/A,"[AI, Computer Science, Information Technology ...","Design, develop, and implement AI / Machine Le...",Required Skills & Qualifications Strong kno...,03/18/2026 13:49:48,https://wuzzuf.net/jobs/p/xvz7wojwcxo4-ai-deve...
523,IT Engineer,N/A,N/A,"Giza, Giza, Egypt",Full Time,Hybrid,"[1, 2]",Entry Level (Junior Level / Fresh Grad),Bachelor's Degree,N/A,"[Information Technology (IT), Engineering Prin...",Unleash Your Engineering Prowess as an Integra...,Exciting Qualifications for a Transformative R...,03/30/2026 12:41:48,https://wuzzuf.net/jobs/p/iys5efvox4as-it-engi...


After:


,job_title,company_name,company_size,location,work_type,work_setting,experience_needed,career_level,education_level,salary,skills_and_tools,job_description,job_requirements,posted_at,url,company_size_min,company_size_max,company_size_avg
383,Senior Document Controller,EMS Solutions,11-50 employees,"6th of October, Giza, Egypt",Full Time,Hybrid,"[5, 10]",Manager,Not Specified,NaN,"[Business Administration, Document Controlling...",Responsibilities: Implement and maintain do...,Bachelor's degree in Business Administration ...,02/26/2026 17:26:44,https://wuzzuf.net/jobs/p/bfasvms4eet5-senior-...,11,50,30
422,Senior Software Engineering Manager-Java,Link Development,501-1000 employees,"Maadi, Cairo, Egypt",Full Time,Hybrid,"[10, 15]",Manager,Bachelor's Degree,NaN,"[Software Engineering, Java, Software Developm...",We are seeking a Senior Software Engineering ...,Proven experience leading teams delivering Ja...,03/09/2026 13:36:04,https://wuzzuf.net/jobs/p/f409jot9zrmg-senior-...,501,1000,750
90,Senior Data Scientist,Egabi FSI,101-500 employees,"Sheraton, Cairo, Egypt",Full Time,On-site,"[3, above_min]",Experienced (Non-Manager),Bachelor's Degree,NaN,"[Computer Science, Data Mining, Data Science, ...","Collect, clean, and preprocess data from vario...","Bachelor’s degree in Computer Science, Data Sc...",03/17/2026 10:39:43,https://wuzzuf.net/jobs/p/o1a7fazksizo-senior-...,101,500,300
472,AI Developer (Outsource),Prometeon tyres group – Egypt,More than 1000 employees,"Alexandria, Egypt",Full Time,On-site,"[0, 2]",Entry Level (Junior Level / Fresh Grad),Bachelor's Degree,NaN,"[AI, Computer Science, Information Technology ...","Design, develop, and implement AI / Machine Le...",Required Skills & Qualifications Strong kno...,03/18/2026 13:49:48,https://wuzzuf.net/jobs/p/xvz7wojwcxo4-ai-deve...,1000,1000,1000
523,IT Engineer,Unknown,Unknown,"Giza, Giza, Egypt",Full Time,Hybrid,"[1, 2]",Entry Level (Junior Level / Fresh Grad),Bachelor's Degree,NaN,"[Information Technology (IT), Engineering Prin...",Unleash Your Engineering Prowess as an Integra...,Exciting Qualifications for a Transformative R...,03/30/2026 12:41:48,https://wuzzuf.net/jobs/p/iys5efvox4as-it-engi...,76,76,76


In [30]:
# ---------- Experience ----------
def extract_experience(x):
    if isinstance(x, list) and len(x) >= 2:
        min_exp = x[0] if isinstance(x[0], (int, float)) else np.nan
        max_exp = x[1] if isinstance(x[1], (int, float)) else np.nan
        return min_exp, max_exp
    return np.nan, np.nan


df[["min_experience_years", "max_experience_years"]] = (
    df["experience_needed"].apply(extract_experience).apply(pd.Series)
)

df.drop(columns="experience_needed", inplace=True)

median_max_exp = df["max_experience_years"].median()
df["max_experience_years"] = df["max_experience_years"].fillna(median_max_exp)

In [31]:
# ---------- Salary ----------
def clean_salary(s):
    if pd.isna(s):
        return np.nan, np.nan, np.nan, np.nan

    s = re.sub(r",?\s*Bonus.*", "", str(s), flags=re.IGNORECASE)

    m = re.search(r"(\d[\d,]*)\s+to\s+(\d[\d,]*)\s+(\w+)\s+Per\s+(\w+)", s)
    if m:
        return (
            float(m.group(1).replace(",", "")),
            float(m.group(2).replace(",", "")),
            m.group(3).upper(),
            m.group(4).capitalize(),
        )

    return np.nan, np.nan, np.nan, np.nan


df[["salary_min", "salary_max", "salary_currency", "salary_period"]] = (
    df["salary"].apply(clean_salary).apply(pd.Series)
)

df.drop(columns="salary", inplace=True)

print(f"Salaries disclosed: {df['salary_min'].notna().sum()} / {len(df)}\n")

Salaries disclosed: 62 / 585



In [32]:
# ---------- Date ----------
df["posted_at"] = pd.to_datetime(
    df["posted_at"],
    format="%m/%d/%Y %H:%M:%S",
    errors="coerce"
)


In [33]:
# ----------  Skills back to list ----------
df["skills_and_tools"] = df["skills_and_tools"].apply(
    lambda x: x if isinstance(x, list) else (
        [s.strip() for s in str(x).split(",") if s.strip()] if pd.notna(x) and x not in ["nan", ""] else []
    )
)

In [34]:
# Work type as list ----------
df["work_type"] = df["work_type"].apply(
    lambda x: [s.strip() for s in str(x).split(",") if s.strip()] if pd.notna(x) and x not in ["Unknown", "nan"] else []
)


In [35]:
# ---------- Categories ----------
translations = {
    "دوام كامل": "Full Time",
    "عمل من مقر الشركة": "On-Site",
    "ذو خبرة (غير إداري)": "Experienced (Non-Manager)",
    "مستوى مبتدئ (مبتدئ / خريج جديد)": "Entry Level (Junior Level / Fresh Grad)",
    "غير محدد": "Not Specified",
    "درجة البكالوريوس": "Bachelor's Degree",
}

# work_type is now a list, handle other cat cols as before
cat_cols = ["work_setting", "career_level", "education_level"]

for col in cat_cols:
    df[col] = df[col].replace(translations)
    df[col] = df[col].str.strip().str.title()
    df[col] = df[col].str.replace(
        r"'([A-Z])", lambda m: "'" + m.group(1).lower(), regex=True
    )

mode_setting = df["work_setting"][df["work_setting"] != "Unknown"].mode()[0]
df["work_setting"] = df["work_setting"].replace("Unknown", mode_setting)


In [36]:
# ---------- Location ----------
def split_location(x):
    parts = [p.strip() for p in str(x).split(",")]
    country = parts[-1] if len(parts) >= 1 else np.nan
    city = parts[-2] if len(parts) >= 2 else np.nan
    return city, country


df[["city", "country"]] = df["location"].apply(split_location).apply(pd.Series)


In [37]:
# ---------- Final ----------
print("Final shape:", df.shape)

Final shape: (585, 24)


In [38]:
# ---------- Save ----------
df.to_csv("clean_data.csv", index=False, encoding="utf-8-sig")

df.to_json(
    "clean_data.json",
    orient="records",
    indent=4,
    force_ascii=False,
    date_format="iso"
)

print("\nDone — files saved to outputs/")



Done — files saved to outputs/
